# SmolLM2 safetensors on PYNQ-Z1
This notebook reads the original full 30-layer checkpoint using the unchanged `src` safetensors loader, then runs the AOT package compiled from that checkpoint on the physical NPU. Compilation and W8A8 quantization happen on the host; the board does not compile the checkpoint. Run all cells in the root PYNQ Python kernel. Change the prompt in the generation cell.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path(os.environ.get("NPU_SMOLLM_DEMO", ".")).resolve()
sys.path.insert(0, str(ROOT))


## Load the original safetensors checkpoint
The source loader memory maps the checkpoint. The compiled package contains the quantized weights used for inference.


In [ ]:
from src.export.core.model_loader import SafeTensors
checkpoint = ROOT / "checkpoint"
with SafeTensors(checkpoint) as tensors:
    print("Loaded safetensors:", len(tensors.keys()), "tensors")
    print("Final norm shape:", tensors["model.norm.weight"].shape)


## Load the compiled model and physical overlay
The repository runtime and vendor driver execute the full compiled model. W8 weights are streamed through CMA; KV cache uses FP32.


In [ ]:
from pynq import Overlay
from src.runtime.core.engine import LLMEngine
overlay = Overlay(str(ROOT / "npu_matrix.bit"))
engine = LLMEngine(ROOT / "package", backend="pynq", overlay=overlay, stream_weights=True)


In [ ]:
prompt = "Write a paragraph about a neural processing unit (NPU)."
max_new_tokens = 32
ids, text, stats = engine.generate(
    prompt, max_new_tokens=max_new_tokens, chat=True,
    on_token=lambda _, piece: print(piece, end="", flush=True),
)
print()
print(json.dumps({"output_ids": ids, "stats": stats.as_dict(), "runtime": engine.package.stats()}, indent=2))


## Release board buffers
Run after generation, or before creating a new engine.

In [ ]:
for buffer in (engine.package.io, engine.package.stage, engine.package.words):
    if buffer is not None:
        buffer.buffer.freebuffer()
del engine
print("Board buffers released")